# Manila Flood Risk Pipeline (PSA 2024)

Unified pipeline: **shp-to-csv → merged-dataset → thesis-ml**

## Required files in `/content/`

| File | Stage |
|------|-------|
| `population-dataset.csv` | merge (PSA 2024 population) |
| `manila_final_base_dataset_NEW.csv` | merge (admin metadata) |
| `MetroManila_Barangay_Flood_Area_Validated.csv` | merge (or run shp stage) |
| `manila_barangay_elevation.csv` | optional |

For **stage shp**, also upload shapefiles to `/content/shapefiles/`:
- `phl_admin4.shp` (+ .dbf, .shx, .prj)
- `MetroManila_Flood_5year.shp`
- `MetroManila_Flood_25year.shp`
- `MetroManila_Flood_100year.shp`

## 1. Get the pipeline script

Upload `manila_flood_risk_pipeline.py` to `/content/`, **or** clone the repo:

In [ ]:
# Option A: clone repo (uncomment and set your repo URL)
# !git clone https://github.com/moridin04/Thesis-Project.git
# %cd Thesis-Project/src/ML-thesis

# Option B: upload manila_flood_risk_pipeline.py to /content/ via Colab file browser
import os
from pathlib import Path

if Path('/content/manila_flood_risk_pipeline.py').exists():
    %cd /content
elif Path('/content/Thesis-Project/src/ML-thesis/manila_flood_risk_pipeline.py').exists():
    %cd /content/Thesis-Project/src/ML-thesis
else:
    print('Upload manila_flood_risk_pipeline.py to /content/ or clone the repo first.')

## 2. (Optional) Mount Google Drive

Use this if your CSV/shapefiles are stored on Drive.

In [ ]:
MOUNT_DRIVE = False  # set True to mount Drive
DRIVE_DATA_DIR = '/content/drive/MyDrive/thesis-data'  # folder with CSVs/shapefiles

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

## 3. Setup & install dependencies

In [ ]:
from manila_flood_risk_pipeline import setup_colab, run_pipeline

paths = setup_colab(
    install_deps=True,
    mount_drive=False,
    drive_data_dir=DRIVE_DATA_DIR if MOUNT_DRIVE else None,
)

## 4. Run pipeline

Choose a stage:
- `'all'` — full pipeline (needs shapefiles)
- `'merge'` — PSA 2024 merge only (needs population + flood CSV)
- `'ml'` — CSI/DPI/ML only (needs ML-ready CSV from merge)

If you already have `MetroManila_Barangay_Flood_Area_Validated.csv`, use `stage='merge'` or set `skip_shp_if_missing=True`.

In [ ]:
STAGE = 'merge'  # 'all', 'shp', 'merge', or 'ml'
SKIP_SHP_IF_MISSING = True  # use existing flood CSV when shapefiles absent

outputs = run_pipeline(
    stage=STAGE,
    data_dir=paths.data_dir,
    shp_dir=paths.shp_dir,
    output_dir=paths.output_dir,
    skip_shp_if_missing=SKIP_SHP_IF_MISSING,
)

list(outputs.keys())

## 5. Run ML stage (if merge was run separately)

In [ ]:
# Uncomment to run ML after merge:
# outputs = run_pipeline(stage='ml', data_dir=paths.data_dir, output_dir=paths.output_dir)
# outputs['final'].sort_values('DPI Score 0to100', ascending=False).head(10)

## 6. Download results

Outputs are saved under `/content/`. Use the Colab file browser (left panel) or:

In [ ]:
from google.colab import files
from pathlib import Path

for name in [
    'manila_ml_ready_dataset.csv',
    'manila_final_DPI_ML_predictions.csv',
    'manila_ml_ready_with_DPI_labels.csv',
    'best_flood_risk_model.pkl',
]:
    p = Path('/content') / name
    if p.exists():
        print(f'Downloading {name}...')
        files.download(str(p))